### RNN - IMDB Sentimewnt Classification

1. Imports

In [15]:
import pandas as pd                     # for loading and handling the CSV dataset
import numpy as np                      # for numerical operations
from sklearn.model_selection import train_test_split   # for splitting train/test sets
from tensorflow.keras.preprocessing.text import Tokenizer         # for converting text to sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences # for padding sequences to equal length
from tensorflow.keras.models import Sequential                    # for building the neural network
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout  # layers used in the RNN model
from tensorflow.keras.optimizers import Adam                      # optimizer for training


2. Load and cleaning Dataset

In [16]:
df = pd.read_csv("IMDB Dataset.csv")    # read the dataset into a pandas DataFrame
df.head()                               # display first few rows to verify structure

# Remove rows where review is NaN or empty
df = df.dropna(subset=['review'])                 # remove NaN reviews
df = df[df['review'].str.strip().str.len() > 0]   # remove empty reviews

# Reset index 
df = df.reset_index(drop=True)


3. Encode Sentiment

In [17]:
# Convert sentiment labels from text to numeric values
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})  # map positive→1, negative→0

4. Train / Test - Slpit

In [18]:
# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df['review'],                       # input text
    df['sentiment'],                    # target labels
    test_size=0.2,                      # 20% test data
    random_state=42                     # ensures reproducibility
)


5. Tokenization

In [19]:
# Tokenize the text reviews
vocab_size = 20000                      # keep only the top 20,000 most frequent words
tokenizer = Tokenizer(num_words=vocab_size)  # create tokenizer object
tokenizer.fit_on_texts(X_train)         # learn word index from training data

# Convert text reviews into sequences of integers
X_train_seq = tokenizer.texts_to_sequences(X_train)  # convert training text to sequences
X_test_seq = tokenizer.texts_to_sequences(X_test)    # convert test text to sequences


6. Padding

In [20]:
# Pad sequences so all reviews have equal length
max_length = 200                        # maximum review length (truncate or pad to 200 words)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)  # pad training sequences
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)    # pad test sequences


7. Build the RNN Model

In [21]:
# Build the RNN model
model = Sequential()                    # initialize a sequential neural network

# Embedding layer converts word IDs into dense vectors
model.add(Embedding(
    input_dim=vocab_size,               # size of vocabulary
    output_dim=128,                     # dimension of embedding vector
    input_length=max_length             # length of each input sequence
))

# LSTM layer learns long-term dependencies in text
model.add(LSTM(128))                    # LSTM with 128 units

# Dropout layer helps prevent overfitting
model.add(Dropout(0.3))                 # randomly drop 30% of neurons during training

# Output layer for binary classification
model.add(Dense(1, activation='sigmoid'))  # sigmoid outputs probability between 0 and 1

# Compile the model with optimizer, loss function, and evaluation metric
model.compile(
    optimizer=Adam(),                   # Adam optimizer
    loss='binary_crossentropy',         # loss for binary classification
    metrics=['accuracy']                # track accuracy during training
)

model.summary()                         # print model architecture


c:\Users\Saad Ehsan\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

8. Train the Model

In [22]:
# Train the model on padded sequences
history = model.fit(
    X_train_pad,                        # training input data
    y_train,                            # training labels
    epochs=5,                           # number of training epochs
    batch_size=128,                     # number of samples per batch
    validation_split=0.2,               # use 20% of training data for validation
    verbose=1                           # show training progress
)


Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 33s 126ms/step - accuracy: 0.7900 - loss: 0.4429 - val_accuracy: 0.8499 - val_loss: 0.3507
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 30s 122ms/step - accuracy: 0.9068 - loss: 0.2484 - val_accuracy: 0.8802 - val_loss: 0.2964
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 30s 121ms/step - accuracy: 0.9410 - loss: 0.1668 - val_accuracy: 0.8785 - val_loss: 0.3279
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 30s 122ms/step - accuracy: 0.9575 - loss: 0.1223 - val_accuracy: 0.8648 - val_loss: 0.3804
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 31s 122ms/step - accuracy: 0.9688 - loss: 0.0946 - val_accuracy: 0.8671 - val_loss: 0.3837


9. Evaluate

In [23]:
# Evaluate model performance on test data
loss, accuracy = model.evaluate(X_test_pad, y_test, verbose=1)  # compute loss and accuracy
print("Test Accuracy:", accuracy)                               # print accuracy score


313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8718 - loss: 0.3623
Test Accuracy: 0.8718000054359436
